# 3 3 count analysis corrected quality

Peer-review copy. Inputs are pseudonymized processed data; see README.md and DATA_DICTIONARY.md. Original preparation/collection steps are outside this package.

In [ ]:
from review_setup import prepare_workspace
prepare_workspace()
from review_setup import initialize_stata
initialize_stata()


## Comparison between higher- and lower-quanlity corrections

#### Daily estimation  [-4,4]

In [ ]:
import pandas as pd
from utils import get_source_corrections, StataDiD

source_corrections = get_source_corrections(window_days=4).query('current_correction==1').dropna(subset=['note_score']).reset_index(drop=True)
source_corrections['treated'] = source_corrections['note_score']>=source_corrections['note_score'].median()
print(f"Daily threshold: {source_corrections['note_score'].median():.3f}")

In [ ]:
ts_posts = pd.read_parquet('data/ts_posts_daily.parquet.gzip').query('-4<=days_from_display<=4').reset_index(drop=True)
ts_posts = ts_posts.merge(source_corrections, on='tweet_id', how='inner')
print(ts_posts.shape)
ts_posts.head()

In [ ]:
stata_did = StataDiD(ts_posts, 'treated', 'tweet_id', 'days_from_display', -4, 4)

In [ ]:
stata_did.multi_did(reg='ppmlhdfe', dev='tweet_count', cluster='account_cluster')

In [ ]:
from utils import get_coefs
means_mp_atts, lls_mp_atts, hls_mp_atts = get_coefs(reg='ppmlhdfe', start_line=8, end_line=15, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
stata_did.tw_did(reg='ppmlhdfe', dev='tweet_count', cluster='account_cluster')

In [ ]:
mean_agg_att, ll_agg_att, hl_agg_att = get_coefs(reg='ppmlhdfe', start_line=2, end_line=3, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
from utils import att_plot
ylims = (-0.262, 0.462)
att_plot(-4, 4, 4, means_mp_atts, lls_mp_atts, hls_mp_atts, mean_agg_att, ll_agg_att, hl_agg_att, 'Daily', 'Days from note display', 'Estimated DiD effect\n(higher vs. lower quality)', 'did_posts_daily_quality', ylims=ylims, legend_ncol=1, legend_title='First correction')

#### Weekly estimation [-4,4]

In [ ]:
source_corrections = get_source_corrections(window_days=28).query('current_correction==1').dropna(subset=['note_score']).reset_index(drop=True)
source_corrections['treated'] = source_corrections['note_score']>=source_corrections['note_score'].median()
print(f"Weekly threshold: {source_corrections['note_score'].median():.3f}")

In [ ]:
ts_posts = pd.read_parquet('data/ts_posts_weekly.parquet.gzip')
ts_posts = ts_posts.merge(source_corrections, on='tweet_id', how='inner')
print(ts_posts.shape)
ts_posts.head()

In [ ]:
stata_did = StataDiD(ts_posts, 'treated', 'tweet_id', 'weeks_from_display', -4, 4)

In [ ]:
stata_did.multi_did(reg='ppmlhdfe', dev='tweet_count', cluster='account_cluster')

In [ ]:
means_mp_atts, lls_mp_atts, hls_mp_atts = get_coefs(reg='ppmlhdfe', start_line=8, end_line=15, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
stata_did.tw_did(reg='ppmlhdfe', dev='tweet_count', cluster='account_cluster')

In [ ]:
mean_agg_att, ll_agg_att, hl_agg_att = get_coefs(reg='ppmlhdfe', start_line=2, end_line=3, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
att_plot(-4, 4, 4, means_mp_atts, lls_mp_atts, hls_mp_atts, mean_agg_att, ll_agg_att, hl_agg_att, 'Weekly', 'Weeks from note display', 'Estimated DiD effect\n(higher vs. lower quality)', 'did_posts_weekly_quality', ylims=ylims, legend_ncol=1, legend_title='First correction')